# CRISP-DM Fase 3: Data Preparation - Validación de Esquema y Quality Gates
## Dataset: Índices Macroeconómicos IPC e IPP Agropecuario (DANE)

**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 3: Data Preparation - Schema Validation) | **Fase PDCO**: DEVELOPMENT  
**Estándares**: Great Expectations Pattern, Pydantic Schema Contracts, ISO/IEC 25010.

---

### 🛡️ 1. Filosofía de Quality Gates
El sistema implementa compuertas de calidad rigurosas bajo el principio **Fail-Fast**:
1. Validación de columnas obligatorias y tipos de datos esperados.
2. Comprobación de restricciones de no negatividad para variables de volumen, precio o precipitación.
3. Evaluación de tasas máximas de nulos permitidas (< 10%).

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


In [ ]:
# ==============================================================================
# [FASE 4: EJECUCIÓN DE QUALITY GATES Y VALIDACIÓN DE CONTRATOS]
# ==============================================================================
from src.validation.schemas import DataValidator

parquet_file = APP_ROOT / "data/processed/dane_ipc.parquet"
df = pd.read_parquet(parquet_file)

# Definición de compuerta de calidad específica para 04_dane_ipc_ipp
primary_col = "unnamed_1"
group_col = "concepto"

required_cols = [col for col in [primary_col, group_col] if col in df.columns]
numeric_cols = [col for col in [primary_col] if col in df.columns and pd.api.types.is_numeric_dtype(df[col])]

print(f"📋 Ejecutando Quality Gate para 'dane_ipc':")
print(f"• Columnas obligatorias evaluadas: {required_cols}")
print(f"• Columnas numéricas restringidas: {numeric_cols}")

is_valid = DataValidator.validate_dataset(
    df=df,
    required_cols=required_cols,
    numeric_cols=numeric_cols
)

if is_valid:
    print("\n✅ QUALITY GATE STATUS: PASSED (Contrato de Datos Aprobado)")
else:
    print("\n❌ QUALITY GATE STATUS: FAILED (Violación de Esquema)")


In [ ]:
# ==============================================================================
# [FASE 4: REPORTE DE CONFORMIDAD Y ASERCIONES FÍSICAS]
# ==============================================================================
conformance_report = []

for col in required_cols:
    null_count = df[col].isnull().sum()
    null_pct = (null_count / len(df)) * 100
    is_ok = null_pct < 10.0
    conformance_report.append({
        "Columna": col,
        "Tipo": str(df[col].dtype),
        "Nulos": null_count,
        "Nulos_%": f"{null_pct:.2f}%",
        "Estado": "OK" if is_ok else "ALERTA"
    })

report_df = pd.DataFrame(conformance_report)
display(report_df)

# Aserción final de integridad
assert not report_df['Estado'].str.contains('FALLO').any(), "Error: Violación crítica de esquema detectada."
print("🎯 Integridad de esquema garantizada para la siguiente etapa.")
